In [1]:
import os
import tempfile
import requests
from scipy.io import loadmat
from datetime import datetime
import re
import subprocess
from urllib.parse import quote
from getpass import getpass


USERNAME = "zahrabah@helsinki.fi"
APP_PASSWORD = "gy8XR-Akg7D-gMQdy-9Tb28-4wdQn"

BASE_URL_VEL = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/raw_velocity_data/"
BASE_URL_BEH = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/behavior_data/"
MATLAB = "/appl/manual_installations/software/matlab/r2024b/bin/matlab"

behList = loadmat("beh_list.mat")['beh_list'][0]
ncsList = loadmat("ncs_list.mat")['ncs_list'][0]
nevList = loadmat("nev_list.mat")['nev_list'][0]


In [2]:
def parse_ncs(fileName):
    ratId = re.search(r"CSC\d+_(?:TN)?(\d+)_", fileName).group(1)
    dateStr = re.search(r"_(\d{6})(?:_|\.ncs)", fileName).group(1)
    return ratId, dateStr

def parse_nev(fileName):
    ratId = re.search(r"Events_(?:TN)?(\d+)_", fileName).group(1)
    dateStr = re.search(r"_(\d{6})(?:_|\.nev)", fileName).group(1)
    return ratId, dateStr

def parse_beh(fileName):
    ratId = re.search(r"Rat[_\s]+(\d+)", fileName).group(1)
    dateStr = re.search(r"(\d{2}-[A-Za-z]{3}-\d{4}|\d{6})",fileName).group(1)
    try:
        dt = datetime.strptime(dateStr, "%d-%b-%Y")
    except ValueError:
        dt = datetime.strptime(dateStr, "%d%m%y")
    date_formatted = dt.strftime("%d%m%y")
    return ratId, date_formatted


In [3]:
def build_file_dict(fileList, parser, label):
    out = {}

    for e in fileList:
        fileName = e[0]

        try:
            ratId, dateStr = parser(fileName)
            key = (ratId, dateStr)
            out[key] = fileName

        except Exception as err:
            print(fileName, "got the following error:", err)

    return out


ncsDict = build_file_dict(ncsList, parse_ncs, "ncs")
nevDict = build_file_dict(nevList, parse_nev, "nev")
behDict = build_file_dict(behList, parse_beh, "beh")

In [4]:
matchingKeys = set(ncsDict) & set(nevDict) & set(behDict)

print("Number of matching file sets:", len(matchingKeys))

Number of matching file sets: 1426


In [6]:
def download_file(base_url, fileName, savePath):
    url = base_url + quote(fileName)
    r = requests.get(url,auth=(USERNAME, APP_PASSWORD))
    r.raise_for_status()
    with open(savePath, "wb") as f:
        f.write(r.content)

In [11]:
for key in behDict:

    ratId, dateStr = key

    behFile = behDict[key]
    nevFile = nevDict[key]
    ncsFile = ncsDict[key]

    print("Processing:", ratId, dateStr)

    with tempfile.TemporaryDirectory() as tmpdir:

        download_file(BASE_URL_BEH, behFile, os.path.join(tmpdir, behFile))
        download_file(BASE_URL_VEL, nevFile, os.path.join(tmpdir, nevFile))
        download_file(BASE_URL_VEL, ncsFile, os.path.join(tmpdir, ncsFile))

        subprocess.run([MATLAB,"-batch",
            f"dataAnalyst('{ratId}', '{dateStr}', '{tmpdir}', '{behFile}', '{nevFile}', '{ncsFile}')"], check=True)

Processing: 10501 011119
Processing: 10501 021119
Processing: 10501 031119
Processing: 10501 041119
Processing: 10501 071119
Processing: 10501 081119


KeyboardInterrupt: 

In [12]:
#Create job list
with open("jobs.txt", "w") as f:
    for key in behDict:
        ratId, dateStr = key

        behFile = behDict[key]
        nevFile = nevDict[key]
        ncsFile = ncsDict[key]

        f.write(f"{ratId}|{dateStr}|{behFile}|{nevFile}|{ncsFile}\n")